# 

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CS675 Amazon Review Analytics") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
from pathlib import Path

raw_data_folder = Path("/home/jovyan/work/data/raw")

for file_path in raw_data_folder.iterdir():
    print(file_path.name, "-", round(file_path.stat().st_size / 1024 / 1024, 1), "MB")

.DS_Store - 0.0 MB
meta_All_Beauty.jsonl - 203.1 MB
All_Beauty.jsonl - 311.5 MB


In [3]:
reviews_path = "/home/jovyan/work/data/raw/All_Beauty.jsonl"

reviews_df = spark.read.json(reviews_path)

reviews_df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [4]:
reviews_df.show(5, truncate=False)

+----------+------------+------+-----------+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-----------------------------------------+----------------------------+-----------------+
|asin      |helpful_vote|images|parent_asin|rating|text                                                                                                                                                                                                                                                                                                        |timestamp    |title                                    |user_id                     |verified_purchase|
+----------+------------+------+-----------+------+-------------------------------------

In [5]:
spark.conf.set("spark.sql.caseSensitive", "true")

In [6]:
metadata_path = "/home/jovyan/work/data/raw/meta_All_Beauty.jsonl"

metadata_df = spark.read.json(metadata_path)

metadata_df.printSchema()

root
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- details: struct (nullable = true)
 |    |-- Action: string (nullable = true)
 |    |-- Active Ingredients: string (nullable = true)
 |    |-- Adjustable Length: string (nullable = true)
 |    |-- Age Range (Description): string (nullable = true)
 |    |-- Age Range Description: string (nullable = true)
 |    |-- Alcohol Content: string (nullable = true)
 |    |-- Allergen Information: string (nullable = true)
 |    |-- Amperage: string (nullable = true)
 |    |-- Antenna: string (nullable = true)
 |    |-- Arch Type: string (nullable = true)
 |    |-- Are Batteries Included: string (nullable = true)
 |    |-- Assembly Required: string (nullable = true)
 |    |-- Assembly Required No: string (nullable = true)


In [7]:
metadata_df.select(
    "parent_asin",
    "title",
    "main_category",
    "price",
    "average_rating",
    "rating_number"
).show(5, truncate=50)



+-----------+--------------------------------------------------+-------------+-----+--------------+-------------+
|parent_asin|                                             title|main_category|price|average_rating|rating_number|
+-----------+--------------------------------------------------+-------------+-----+--------------+-------------+
| B01CUPMQZE|Howard LC0008 Leather Conditioner, 8-Ounce (4-P...|   All Beauty| NULL|           4.8|           10|
| B076WQZGPM|Yes to Tomatoes Detoxifying Charcoal Cleanser (...|   All Beauty| NULL|           4.5|            3|
| B000B658RI|  Eye Patch Black Adult with Tie Band (6 Per Pack)|   All Beauty| NULL|           4.4|           26|
| B088FKY3VD|Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4D...|   All Beauty| NULL|           3.1|          102|
| B07NGFDN6G|Precision Plunger Bars for Cartridge Grips – 93...|   All Beauty| NULL|           4.3|            7|
+-----------+--------------------------------------------------+-------------+-----+----

In [8]:
reviews_df.select(
    "parent_asin",
    "rating",
    "title",
    "helpful_vote",
    "verified_purchase",
    "timestamp"
).show(10, truncate=40)

+-----------+------+----------------------------------------+------------+-----------------+-------------+
|parent_asin|rating|                                   title|helpful_vote|verified_purchase|    timestamp|
+-----------+------+----------------------------------------+------------+-----------------+-------------+
| B00YQ6X8EO|   5.0|Such a lovely scent but not overpower...|           0|             true|1588687728923|
| B081TJ8YS3|   4.0|  Works great but smells a little weird.|           1|             true|1588615855070|
| B097R46CSY|   5.0|                                    Yes!|           2|             true|1589665266052|
| B09JS339BZ|   1.0|                       Synthetic feeling|           0|             true|1643393630220|
| B08BZ63GMJ|   5.0|                                      A+|           0|             true|1609322563534|
| B00R8DXL44|   4.0|                            Pretty Color|           0|             true|1598567408138|
| B099DRHW5V|   5.0|                 

In [9]:
#see if the join key exists
print("Reviews has parent_asin:", "parent_asin" in reviews_df.columns)
print("Metadata has parent_asin:", "parent_asin" in metadata_df.columns)

Reviews has parent_asin: True
Metadata has parent_asin: True


In [10]:
#column pruning - leave only the useful columns for analysis

reviews_clean_df = reviews_df.select(
    "parent_asin",
    "asin",
    "user_id",
    "rating",
    "title",
    "text",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
)

metadata_clean_df = metadata_df.select(
    "parent_asin",
    "title",
    "main_category",
    "price",
    "average_rating",
    "rating_number",
    "store"
)

In [11]:
# investigate price missing issue

from pyspark.sql import functions as F

price_summary = metadata_clean_df.select(
    F.count("*").alias("total_products"),
    F.sum(F.when(F.col("price").isNull(), 1).otherwise(0)).alias("missing_price"),
    F.sum(F.when(F.col("price").isNotNull(), 1).otherwise(0)).alias("available_price")
).withColumn(
    "missing_price_percent",
    F.round(F.col("missing_price") / F.col("total_products") * 100, 2)
)

price_summary.show()

+--------------+-------------+---------------+---------------------+
|total_products|missing_price|available_price|missing_price_percent|
+--------------+-------------+---------------+---------------------+
|        112590|        94886|          17704|                84.28|
+--------------+-------------+---------------+---------------------+



In [12]:
metadata_clean_df.select("price").describe().show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|             17704|
|   mean|27.255729778581074|
| stddev| 50.47201982517162|
|    min|              0.01|
|    max|           2548.98|
+-------+------------------+



In [13]:
metadata_clean_df.filter(
    F.col("price").isNotNull() & (F.col("price") <= 0)
).select("parent_asin", "title", "price").show(20, truncate=50)

+-----------+-----+-----+
|parent_asin|title|price|
+-----------+-----+-----+
+-----------+-----+-----+



In [14]:
metadata_clean_df = metadata_clean_df.withColumn(
    "price_missing",
    F.col("price").isNull()
)

In [15]:
#create price_missing flag

metadata_clean_df.select(
    "parent_asin",
    "title",
    "price",
    "price_missing"
).show(10, truncate=40)

+-----------+----------------------------------------+-----+-------------+
|parent_asin|                                   title|price|price_missing|
+-----------+----------------------------------------+-----+-------------+
| B01CUPMQZE|Howard LC0008 Leather Conditioner, 8-...| NULL|         true|
| B076WQZGPM|Yes to Tomatoes Detoxifying Charcoal ...| NULL|         true|
| B000B658RI|Eye Patch Black Adult with Tie Band (...| NULL|         true|
| B088FKY3VD|Tattoo Eyebrow Stickers, Waterproof E...| NULL|         true|
| B07NGFDN6G|Precision Plunger Bars for Cartridge ...| NULL|         true|
| B07G9GWFSM|Lurrose 100Pcs Full Cover Fake Toenai...| 6.99|        false|
| B08XZ97HFY|Stain Bonnet For Baby Bonnet Silk Sle...| NULL|         true|
| B08DNQTTQK|50 Pieces False Eyelash Packaging Box...| NULL|         true|
| B01ERJEGS6|              Gold extatic Musk EDT 90ml|86.95|        false|
| B08P7LXKP7|4 Pieces Satin Bonnet Adjustable Slee...| NULL|         true|
+-----------+------------

In [16]:
#check for missing values in core columns

review_missing_summary = reviews_clean_df.select(
    F.count("*").alias("total_reviews"),
    F.sum(F.when(F.col("parent_asin").isNull(), 1).otherwise(0)).alias("missing_parent_asin"),
    F.sum(F.when(F.col("rating").isNull(), 1).otherwise(0)).alias("missing_rating"),
    F.sum(F.when(F.col("text").isNull(), 1).otherwise(0)).alias("missing_text"),
    F.sum(F.when(F.col("timestamp").isNull(), 1).otherwise(0)).alias("missing_timestamp"),
    F.sum(F.when(F.col("helpful_vote").isNull(), 1).otherwise(0)).alias("missing_helpful_vote"),
    F.sum(F.when(F.col("verified_purchase").isNull(), 1).otherwise(0)).alias("missing_verified_purchase")
)

review_missing_summary.show()

+-------------+-------------------+--------------+------------+-----------------+--------------------+-------------------------+
|total_reviews|missing_parent_asin|missing_rating|missing_text|missing_timestamp|missing_helpful_vote|missing_verified_purchase|
+-------------+-------------------+--------------+------------+-----------------+--------------------+-------------------------+
|       701528|                  0|             0|           0|                0|                   0|                        0|
+-------------+-------------------+--------------+------------+-----------------+--------------------+-------------------------+



In [17]:
reviews_clean_df.groupBy("rating") \
    .count() \
    .orderBy("rating") \
    .show()

+------+------+
|rating| count|
+------+------+
|   1.0|102080|
|   2.0| 43034|
|   3.0| 56307|
|   4.0| 79381|
|   5.0|420726|
+------+------+



In [18]:
#check for duplicate reviews

total_reviews = reviews_clean_df.count()

unique_reviews = reviews_clean_df.dropDuplicates().count()

print("Total reviews:", total_reviews)
print("Unique reviews:", unique_reviews)
print("Duplicate rows:", total_reviews - unique_reviews)

Total reviews: 701528
Unique reviews: 694253
Duplicate rows: 7275


In [19]:
duplicate_key_summary = reviews_clean_df.groupBy(
    "user_id",
    "asin",
    "timestamp"
).count().filter(
    F.col("count") > 1
)

print("Duplicate review keys:", duplicate_key_summary.count())

Duplicate review keys: 6139


In [20]:
# remove duplicate reviews

reviews_clean_df = reviews_clean_df.dropDuplicates()

print("Rows after removing exact duplicates:", reviews_clean_df.count())

Rows after removing exact duplicates: 694253


In [21]:
duplicate_key_summary.orderBy(F.desc("count")).show(20, truncate=False)

+----------------------------+----------+-------------+-----+
|user_id                     |asin      |timestamp    |count|
+----------------------------+----------+-------------+-----+
|AHK3S32VWSXOBIYYF2MKGXUDRDBA|B008FR532U|1643563631330|10   |
|AHUWZSKOOZ475MPMYUKUDQJQTYVQ|B07CHB69Y3|1545362378685|8    |
|AHC4WQGKGVP5UQ2FRGWY6TWAOETA|B074W1M23V|1628341545944|8    |
|AHJEGTHANR23UG3PMPA6MMCPV3RA|B0873Y1WFZ|1633280872333|8    |
|AEHUG4KT7XXVNRIUYLKMVM576CXQ|B078919VBD|1692582159574|7    |
|AFVVSP4QRQCORIAEG4V4PIJQOXGA|B017O89LD2|1534040396015|7    |
|AG2MWE5FOYGZUXGKFDD2LBRWY4PQ|B00Q1BNG6C|1421927031000|7    |
|AHWC5CGKFYDUDAVDUL2654BKBX6A|B07C6BDC2Q|1534029982664|7    |
|AGICMDY5MI55SBAEKWZNKMQX3TDQ|B014E2CSIQ|1546099540658|7    |
|AEFYI6LZU6TKJZ32LBOFO4CI2ZNA|B0798S8C41|1525653377354|7    |
|AEZS34HKB5WZQHHJQ2NZQZ5PFZJA|B087PY8ZCQ|1595479486366|7    |
|AHVU665DOVO3YQR3EPBVHZNRPGVQ|B01F8P1TI8|1509695850015|6    |
|AHGWGX6GLP7IGU6NZC77PFASS6OQ|B07M9ZSMJW|1564771673741|6    |
|AHPDJAE

In [22]:
sample_duplicate = duplicate_key_summary.orderBy(F.desc("count")).first()

reviews_df.filter(
    (F.col("user_id") == sample_duplicate["user_id"]) &
    (F.col("asin") == sample_duplicate["asin"]) &
    (F.col("timestamp") == sample_duplicate["timestamp"])
).select(
    "user_id",
    "asin",
    "timestamp",
    "rating",
    "title",
    "text",
    "helpful_vote",
    "verified_purchase"
).show(truncate=False)

+----------------------------+----------+-------------+------+------------------------------+---------------------+------------+-----------------+
|user_id                     |asin      |timestamp    |rating|title                         |text                 |helpful_vote|verified_purchase|
+----------------------------+----------+-------------+------+------------------------------+---------------------+------------+-----------------+
|AHK3S32VWSXOBIYYF2MKGXUDRDBA|B008FR532U|1643563631330|5.0   |Produces natural glow on face.|Evens out complexion.|0           |true             |
|AHK3S32VWSXOBIYYF2MKGXUDRDBA|B008FR532U|1643563631330|5.0   |Produces natural glow on face.|Evens out complexion.|0           |true             |
|AHK3S32VWSXOBIYYF2MKGXUDRDBA|B008FR532U|1643563631330|5.0   |Produces natural glow on face.|Evens out complexion.|0           |true             |
|AHK3S32VWSXOBIYYF2MKGXUDRDBA|B008FR532U|1643563631330|5.0   |Produces natural glow on face.|Evens out complexion.|0  

In [23]:
# clean timestamp data
reviews_clean_df = reviews_clean_df.withColumn(
    "review_timestamp",
    F.to_timestamp(
        F.from_unixtime(F.col("timestamp") / 1000)
    )
)

In [24]:
reviews_clean_df = reviews_clean_df \
    .withColumn("review_date", F.to_date("review_timestamp")) \
    .withColumn("review_year", F.year("review_timestamp")) \
    .withColumn("review_month", F.month("review_timestamp"))

In [25]:
reviews_clean_df.select(
    "timestamp",
    "review_timestamp",
    "review_date",
    "review_year",
    "review_month"
).show(10, truncate=False)

+-------------+-------------------+-----------+-----------+------------+
|timestamp    |review_timestamp   |review_date|review_year|review_month|
+-------------+-------------------+-----------+-----------+------------+
|1662827578220|2022-09-10 16:32:58|2022-09-10 |2022       |9           |
|1544671363612|2018-12-13 03:22:43|2018-12-13 |2018       |12          |
|1596927321037|2020-08-08 22:55:21|2020-08-08 |2020       |8           |
|1607908220469|2020-12-14 01:10:20|2020-12-14 |2020       |12          |
|1623767406253|2021-06-15 14:30:06|2021-06-15 |2021       |6           |
|1597830191303|2020-08-19 09:43:11|2020-08-19 |2020       |8           |
|1637249892882|2021-11-18 15:38:12|2021-11-18 |2021       |11          |
|1582579746910|2020-02-24 21:29:06|2020-02-24 |2020       |2           |
|1550641088313|2019-02-20 05:38:08|2019-02-20 |2019       |2           |
|1577421521188|2019-12-27 04:38:41|2019-12-27 |2019       |12          |
+-------------+-------------------+-----------+----

In [26]:
#check for suspicious date data

reviews_clean_df.select(
    F.min("review_date").alias("earliest_review"),
    F.max("review_date").alias("latest_review")
).show()

+---------------+-------------+
|earliest_review|latest_review|
+---------------+-------------+
|     2000-11-01|   2023-09-09|
+---------------+-------------+



In [27]:
reviews_clean_df.groupBy("review_year") \
    .count() \
    .orderBy("review_year") \
    .show(50)

+-----------+------+
|review_year| count|
+-----------+------+
|       2000|     1|
|       2001|    11|
|       2002|    24|
|       2003|    56|
|       2004|   130|
|       2005|   265|
|       2006|   416|
|       2007|  1150|
|       2008|  1251|
|       2009|  1198|
|       2010|  1568|
|       2011|  1855|
|       2012|  2774|
|       2013|  7882|
|       2014| 16107|
|       2015| 35854|
|       2016| 62069|
|       2017| 67389|
|       2018| 72008|
|       2019| 98148|
|       2020|125428|
|       2021|123512|
|       2022| 60945|
|       2023| 14212|
+-----------+------+



In [28]:
reviews_clean_df = reviews_clean_df.withColumn(
    "review_text_length",
    F.length(F.col("text"))
)

In [29]:
#review text length

reviews_clean_df.select(
    "rating",
    "text",
    "review_text_length"
).show(10, truncate=50)

+------+--------------------------------------------------+------------------+
|rating|                                              text|review_text_length|
+------+--------------------------------------------------+------------------+
|   1.0|I read the description and thought this was som...|               532|
|   4.0|This flat iron works well to straighten hair, b...|               210|
|   5.0|I like to keep extra spray bottles on hand. The...|               452|
|   5.0|I like it. I'll use nearly anything so long as ...|               754|
|   5.0|In this jar you get 30 pairs of marine algae ey...|               676|
|   4.0|Though a bit pricey at $20 for 3.4 ounces, this...|               326|
|   3.0|Since there aren’t any reviews for this product...|              1729|
|   5.0|and before moisturizing This serum helps fight ...|               289|
|   5.0|Great soap! Smell is not too strong and lathers...|                55|
|   5.0|       Great scent. Nice face feel. Will buy

In [30]:
reviews_clean_df.select(
    F.min("helpful_vote").alias("min_helpful_vote"),
    F.max("helpful_vote").alias("max_helpful_vote"),
    F.avg("helpful_vote").alias("avg_helpful_vote")
).show()

+----------------+----------------+------------------+
|min_helpful_vote|max_helpful_vote|  avg_helpful_vote|
+----------------+----------------+------------------+
|               0|             646|0.9241731760611765|
+----------------+----------------+------------------+



In [31]:
reviews_clean_df.groupBy("verified_purchase") \
    .count() \
    .show()

+-----------------+------+
|verified_purchase| count|
+-----------------+------+
|             true|628456|
|            false| 65797|
+-----------------+------+



In [32]:
helpful_percentiles = reviews_clean_df.approxQuantile(
    "helpful_vote",
    [0.50, 0.75, 0.90, 0.95, 0.99],
    0.01
)

print("Median:", helpful_percentiles[0])
print("75th percentile:", helpful_percentiles[1])
print("90th percentile:", helpful_percentiles[2])
print("95th percentile:", helpful_percentiles[3])
print("99th percentile:", helpful_percentiles[4])

Median: 0.0
75th percentile: 1.0
90th percentile: 2.0
95th percentile: 3.0
99th percentile: 646.0


In [33]:
helpful_vote_summary = reviews_clean_df.select(
    F.count("*").alias("total_reviews"),
    F.sum(
        F.when(F.col("helpful_vote") == 0, 1).otherwise(0)
    ).alias("zero_helpful_votes"),
    F.sum(
        F.when(F.col("helpful_vote") > 0, 1).otherwise(0)
    ).alias("positive_helpful_votes")
).withColumn(
    "positive_helpful_percent",
    F.round(
        F.col("positive_helpful_votes") / F.col("total_reviews") * 100,
        2
    )
)

helpful_vote_summary.show()

+-------------+------------------+----------------------+------------------------+
|total_reviews|zero_helpful_votes|positive_helpful_votes|positive_helpful_percent|
+-------------+------------------+----------------------+------------------------+
|       694253|            508729|                185524|                   26.72|
+-------------+------------------+----------------------+------------------------+



In [34]:
helpful_percentiles_accurate = reviews_clean_df.approxQuantile(
    "helpful_vote",
    [0.95, 0.99, 0.995, 0.999],
    0.0001
)

print("95th percentile:", helpful_percentiles_accurate[0])
print("99th percentile:", helpful_percentiles_accurate[1])
print("99.5th percentile:", helpful_percentiles_accurate[2])
print("99.9th percentile:", helpful_percentiles_accurate[3])

95th percentile: 4.0
99th percentile: 13.0
99.5th percentile: 21.0
99.9th percentile: 59.0


In [35]:
reviews_clean_df = reviews_clean_df.withColumn(
    "received_helpful_vote",
    F.when(F.col("helpful_vote") > 0, 1).otherwise(0)
)

In [36]:
reviews_clean_df = reviews_clean_df.withColumn(
    "log_helpful_vote",
    F.log1p(F.col("helpful_vote"))
)

In [37]:
#compare verified vs unverified reviews
verified_comparison = reviews_clean_df.groupBy("verified_purchase").agg(
    F.count("*").alias("review_count"),
    F.round(F.avg("rating"), 3).alias("avg_rating"),
    F.round(F.avg("review_text_length"), 2).alias("avg_text_length"),
    F.round(F.avg("helpful_vote"), 3).alias("avg_helpful_vote"),
    F.round(F.avg("received_helpful_vote") * 100, 2).alias("percent_with_helpful_vote")
)

verified_comparison.show()

+-----------------+------------+----------+---------------+----------------+-------------------------+
|verified_purchase|review_count|avg_rating|avg_text_length|avg_helpful_vote|percent_with_helpful_vote|
+-----------------+------------+----------+---------------+----------------+-------------------------+
|             true|      628456|     3.949|         152.55|           0.906|                    26.56|
|            false|       65797|     4.075|         369.43|           1.102|                    28.29|
+-----------------+------------+----------+---------------+----------------+-------------------------+



In [38]:
reviews_clean_df.groupBy(
    "verified_purchase",
    "rating"
).count().orderBy(
    "verified_purchase",
    "rating"
).show()

+-----------------+------+------+
|verified_purchase|rating| count|
+-----------------+------+------+
|            false|   1.0|  7566|
|            false|   2.0|  3265|
|            false|   3.0|  5161|
|            false|   4.0| 10466|
|            false|   5.0| 39339|
|             true|   1.0| 93322|
|             true|   2.0| 39336|
|             true|   3.0| 50559|
|             true|   4.0| 68142|
|             true|   5.0|377097|
+-----------------+------+------+



In [39]:
from pyspark.sql.window import Window

rating_by_verified = reviews_clean_df.groupBy(
    "verified_purchase",
    "rating"
).count()

verified_window = Window.partitionBy("verified_purchase")

rating_by_verified = rating_by_verified.withColumn(
    "group_total",
    F.sum("count").over(verified_window)
).withColumn(
    "rating_percent",
    F.round(F.col("count") / F.col("group_total") * 100, 2)
)

rating_by_verified.orderBy(
    "verified_purchase",
    "rating"
).show()

+-----------------+------+------+-----------+--------------+
|verified_purchase|rating| count|group_total|rating_percent|
+-----------------+------+------+-----------+--------------+
|            false|   1.0|  7566|      65797|          11.5|
|            false|   2.0|  3265|      65797|          4.96|
|            false|   3.0|  5161|      65797|          7.84|
|            false|   4.0| 10466|      65797|         15.91|
|            false|   5.0| 39339|      65797|         59.79|
|             true|   1.0| 93322|     628456|         14.85|
|             true|   2.0| 39336|     628456|          6.26|
|             true|   3.0| 50559|     628456|          8.04|
|             true|   4.0| 68142|     628456|         10.84|
|             true|   5.0|377097|     628456|          60.0|
+-----------------+------+------+-----------+--------------+



In [40]:
#verified status by year
reviews_clean_df.groupBy(
    "review_year",
    "verified_purchase"
).count().orderBy(
    "review_year",
    "verified_purchase"
).show(100)

+-----------+-----------------+------+
|review_year|verified_purchase| count|
+-----------+-----------------+------+
|       2000|            false|     1|
|       2001|            false|    11|
|       2002|            false|    20|
|       2002|             true|     4|
|       2003|            false|    41|
|       2003|             true|    15|
|       2004|            false|    94|
|       2004|             true|    36|
|       2005|            false|   176|
|       2005|             true|    89|
|       2006|            false|   246|
|       2006|             true|   170|
|       2007|            false|   600|
|       2007|             true|   550|
|       2008|            false|   662|
|       2008|             true|   589|
|       2009|            false|   499|
|       2009|             true|   699|
|       2010|            false|   357|
|       2010|             true|  1211|
|       2011|            false|   429|
|       2011|             true|  1426|
|       2012|            

In [41]:
#compare verified and unverified reviews within each year
verified_by_year = reviews_clean_df.groupBy(
    "review_year",
    "verified_purchase"
).agg(
    F.count("*").alias("review_count"),
    F.round(F.avg("rating"), 3).alias("avg_rating"),
    F.round(F.avg("review_text_length"), 2).alias("avg_text_length"),
    F.round(F.avg("received_helpful_vote") * 100, 2)
        .alias("percent_with_helpful_vote")
)

verified_by_year.orderBy(
    "review_year",
    "verified_purchase"
).show(100)

+-----------+-----------------+------------+----------+---------------+-------------------------+
|review_year|verified_purchase|review_count|avg_rating|avg_text_length|percent_with_helpful_vote|
+-----------+-----------------+------------+----------+---------------+-------------------------+
|       2000|            false|           1|       5.0|          239.0|                    100.0|
|       2001|            false|          11|     4.636|        1676.45|                    100.0|
|       2002|            false|          20|       4.4|          907.3|                     90.0|
|       2002|             true|           4|      4.25|         292.75|                    100.0|
|       2003|            false|          41|     4.317|         953.54|                    100.0|
|       2003|             true|          15|       4.4|         1018.0|                    100.0|
|       2004|            false|          94|     4.032|         978.04|                    96.81|
|       2004|       

In [42]:
year_window = Window.partitionBy("review_year")

verified_share_by_year = reviews_clean_df.groupBy(
    "review_year",
    "verified_purchase"
).count().withColumn(
    "year_total",
    F.sum("count").over(year_window)
).withColumn(
    "percent_of_year",
    F.round(F.col("count") / F.col("year_total") * 100, 2)
)

verified_share_by_year.orderBy(
    "review_year",
    "verified_purchase"
).show(100)

+-----------+-----------------+------+----------+---------------+
|review_year|verified_purchase| count|year_total|percent_of_year|
+-----------+-----------------+------+----------+---------------+
|       2000|            false|     1|         1|          100.0|
|       2001|            false|    11|        11|          100.0|
|       2002|            false|    20|        24|          83.33|
|       2002|             true|     4|        24|          16.67|
|       2003|            false|    41|        56|          73.21|
|       2003|             true|    15|        56|          26.79|
|       2004|            false|    94|       130|          72.31|
|       2004|             true|    36|       130|          27.69|
|       2005|            false|   176|       265|          66.42|
|       2005|             true|    89|       265|          33.58|
|       2006|            false|   246|       416|          59.13|
|       2006|             true|   170|       416|          40.87|
|       20

In [43]:
# helpfulness analysis controled for review age 
reviews_clean_df = reviews_clean_df.withColumn(
    "review_age_days",
    F.datediff(
        F.lit("2023-09-09"),
        F.col("review_date")
    )
)

In [44]:
reviews_clean_df = reviews_clean_df.withColumn(
    "helpful_votes_per_year",
    F.when(
        F.col("review_age_days") > 0,
        F.col("helpful_vote") / (F.col("review_age_days") / 365.25)
    ).otherwise(0)
)

In [45]:
recent_verified_comparison = reviews_clean_df.filter(
    (F.col("review_year") >= 2018) &
    (F.col("review_year") <= 2022)
).groupBy(
    "verified_purchase"
).agg(
    F.count("*").alias("review_count"),
    F.round(F.avg("rating"), 3).alias("avg_rating"),
    F.round(F.avg("review_text_length"), 2).alias("avg_text_length"),
    F.round(F.avg("received_helpful_vote") * 100, 2)
        .alias("percent_with_helpful_vote"),
    F.round(F.avg("helpful_votes_per_year"), 3)
        .alias("avg_helpful_votes_per_year")
)

recent_verified_comparison.show()

+-----------------+------------+----------+---------------+-------------------------+--------------------------+
|verified_purchase|review_count|avg_rating|avg_text_length|percent_with_helpful_vote|avg_helpful_votes_per_year|
+-----------------+------------+----------+---------------+-------------------------+--------------------------+
|             true|      444015|     3.903|         146.15|                    24.02|                     0.281|
|            false|       36026|     3.956|         330.98|                    22.77|                     0.232|
+-----------------+------------+----------+---------------+-------------------------+--------------------------+



In [46]:
reviews_for_helpfulness_df = reviews_clean_df.filter(
    F.col("review_age_days") >= 365
)

In [47]:
#test whether longer reviews are more likely to receive helpful votes
reviews_clean_df = reviews_clean_df.withColumn(
    "text_length_bin",
    F.when(F.col("review_text_length") < 50, "Under 50")
     .when(F.col("review_text_length") < 150, "50-149")
     .when(F.col("review_text_length") < 300, "150-299")
     .when(F.col("review_text_length") < 600, "300-599")
     .otherwise("600+")
)

In [48]:
length_helpfulness = reviews_clean_df.filter(
    (F.col("review_year") >= 2018) &
    (F.col("review_year") <= 2022)
).groupBy(
    "text_length_bin"
).agg(
    F.count("*").alias("review_count"),
    F.round(F.avg("rating"), 3).alias("avg_rating"),
    F.round(F.avg("received_helpful_vote") * 100, 2)
        .alias("percent_with_helpful_vote"),
    F.round(F.avg("helpful_votes_per_year"), 3)
        .alias("avg_helpful_votes_per_year")
)

length_helpfulness.orderBy(
    F.when(F.col("text_length_bin") == "Under 50", 1)
     .when(F.col("text_length_bin") == "50-149", 2)
     .when(F.col("text_length_bin") == "150-299", 3)
     .when(F.col("text_length_bin") == "300-599", 4)
     .otherwise(5)
).show()

+---------------+------------+----------+-------------------------+--------------------------+
|text_length_bin|review_count|avg_rating|percent_with_helpful_vote|avg_helpful_votes_per_year|
+---------------+------------+----------+-------------------------+--------------------------+
|       Under 50|      134618|     4.127|                    13.18|                     0.094|
|         50-149|      178296|     3.844|                    20.58|                     0.186|
|        150-299|      102189|     3.776|                    30.38|                     0.363|
|        300-599|       48885|     3.809|                    42.65|                      0.63|
|           600+|       16053|       3.9|                    53.16|                     1.204|
+---------------+------------+----------+-------------------------+--------------------------+



In [49]:
#imputation
metadata_missing_summary = metadata_clean_df.select(
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in [
            "title",
            "main_category",
            "price",
            "average_rating",
            "rating_number",
            "store"
        ]
    ]
)

metadata_missing_summary.show()

+-----+-------------+-----+--------------+-------------+-----+
|title|main_category|price|average_rating|rating_number|store|
+-----+-------------+-----+--------------+-------------+-----+
|    0|            0|94886|             0|            0|11331|
+-----+-------------+-----+--------------+-------------+-----+



In [50]:
#outlier 
metadata_clean_df = metadata_clean_df \
    .withColumn(
        "store_missing",
        F.col("store").isNull()
    ) \
    .withColumn(
        "store_imputed",
        F.coalesce(F.col("store"), F.lit("Unknown"))
    )

In [51]:
reviews_clean_df = reviews_clean_df.withColumn(
    "helpful_vote_capped",
    F.least(F.col("helpful_vote"), F.lit(13))
)

In [52]:
reviews_clean_df.select(
    F.max("helpful_vote").alias("original_max"),
    F.max("helpful_vote_capped").alias("capped_max"),
    F.round(F.avg("helpful_vote"), 3).alias("original_mean"),
    F.round(F.avg("helpful_vote_capped"), 3).alias("capped_mean")
).show()

+------------+----------+-------------+-----------+
|original_max|capped_max|original_mean|capped_mean|
+------------+----------+-------------+-----------+
|         646|        13|        0.924|      0.716|
+------------+----------+-------------+-----------+



In [53]:
#normalization

length_stats = reviews_clean_df.select(
    F.avg("review_text_length").alias("mean_length"),
    F.stddev("review_text_length").alias("std_length")
).first()

mean_length = length_stats["mean_length"]
std_length = length_stats["std_length"]

reviews_clean_df = reviews_clean_df.withColumn(
    "review_text_length_z",
    (
        F.col("review_text_length") - F.lit(mean_length)
    ) / F.lit(std_length)
)

In [54]:
reviews_clean_df.select(
    F.round(F.avg("review_text_length_z"), 3).alias("z_score_mean"),
    F.round(F.stddev("review_text_length_z"), 3).alias("z_score_stddev")
).show()

+------------+--------------+
|z_score_mean|z_score_stddev|
+------------+--------------+
|         0.0|           1.0|
+------------+--------------+



In [55]:
#encoding
reviews_clean_df = reviews_clean_df.withColumn(
    "verified_purchase_encoded",
    F.when(F.col("verified_purchase"), 1).otherwise(0)
)

In [56]:
reviews_clean_df.select(
    "verified_purchase",
    "verified_purchase_encoded"
).distinct().show()

+-----------------+-------------------------+
|verified_purchase|verified_purchase_encoded|
+-----------------+-------------------------+
|             true|                        1|
|            false|                        0|
+-----------------+-------------------------+



## Preprocessing Decisions

- **Imputation:** Missing `store` values were replaced with `"Unknown"`.
  A separate `store_missing` indicator was retained so that missingness was
  not hidden.

- **Price missingness:** Price was missing for 84.28% of products. It was not
  imputed because replacing most values with an estimate could substantially
  distort product-level analysis. Instead, a `price_missing` flag was created.

- **Outlier treatment:** `helpful_vote` was highly right-skewed, with a maximum
  of 646 and a 99th percentile of 13. A capped version was created at 13 while
  preserving the original column.

- **Normalization:** Review-text length was standardized using a z-score. The
  transformed variable had a mean of approximately 0 and a standard deviation
  of approximately 1.

- **Encoding:** `verified_purchase` was converted from Boolean values to a
  numeric indicator where true = 1 and false = 0.

- **Binning:** Review length and product popularity were grouped into
  interpretable ranges for comparison.

In [57]:
#join reviews with product metadata
metadata_for_join_df = metadata_clean_df.select(
    "parent_asin",
    F.col("title").alias("product_title"),
    "main_category",
    "price",
    "price_missing",
    "average_rating",
    "rating_number",
    F.col("store_imputed").alias("store")
)

In [58]:
joined_df = reviews_clean_df.join(
    metadata_for_join_df,
    on="parent_asin",
    how="inner"
)

In [59]:
print("Clean review rows:", reviews_clean_df.count())
print("Metadata rows:", metadata_for_join_df.count())
print("Joined rows:", joined_df.count())

Clean review rows: 694253
Metadata rows: 112590
Joined rows: 694253


In [60]:
joined_df.select(
    "parent_asin",
    "product_title",
    "store",
    "rating",
    "verified_purchase",
    "review_text_length",
    "helpful_vote",
    "rating_number"
).show(10, truncate=40)

+-----------+----------------------------------------+----------+------+-----------------+------------------+------------+-------------+
|parent_asin|                           product_title|     store|rating|verified_purchase|review_text_length|helpful_vote|rating_number|
+-----------+----------------------------------------+----------+------+-----------------+------------------+------------+-------------+
| B0B7RBK4NJ|Face Painting Kits with Stencils for ...|  FROHOUSE|   1.0|            false|               532|           0|            4|
| B07F8PCLDV|Hair Straightener, Flat Iron for Hair...|    COOQIN|   4.0|            false|               210|           0|           36|
| B08BNNNMH5|sdoot Misting Spray Bottle, 2 Pack 20...|     sdoot|   5.0|            false|               452|           2|           80|
| B07YNDWRCB|Hylunia Hydrate Body Wash - Energizin...|   Hylunia|   5.0|            false|               754|           1|           30|
| B08KWN77LW|GENSKIN Generation Skin Hydr

In [61]:
joined_df = joined_df.withColumn(
    "product_popularity_bin",
    F.when(F.col("rating_number") < 10, "Under 10 ratings")
     .when(F.col("rating_number") < 50, "10-49 ratings")
     .when(F.col("rating_number") < 200, "50-199 ratings")
     .when(F.col("rating_number") < 1000, "200-999 ratings")
     .otherwise("1000+ ratings")
)

In [62]:
popularity_analysis = joined_df.groupBy(
    "product_popularity_bin"
).agg(
    F.count("*").alias("review_count"),
    F.round(F.avg("rating"), 3).alias("avg_review_rating"),
    F.round(F.avg("review_text_length"), 2).alias("avg_text_length"),
    F.round(F.avg("received_helpful_vote") * 100, 2)
        .alias("percent_with_helpful_vote"),
    F.round(F.avg("helpful_votes_per_year"), 3)
        .alias("avg_helpful_votes_per_year")
)

In [63]:
popularity_analysis.orderBy(
    F.when(F.col("product_popularity_bin") == "Under 10 ratings", 1)
     .when(F.col("product_popularity_bin") == "10-49 ratings", 2)
     .when(F.col("product_popularity_bin") == "50-199 ratings", 3)
     .when(F.col("product_popularity_bin") == "200-999 ratings", 4)
     .otherwise(5)
).show()

+----------------------+------------+-----------------+---------------+-------------------------+--------------------------+
|product_popularity_bin|review_count|avg_review_rating|avg_text_length|percent_with_helpful_vote|avg_helpful_votes_per_year|
+----------------------+------------+-----------------+---------------+-------------------------+--------------------------+
|      Under 10 ratings|       89715|            3.789|         170.63|                    21.96|                     0.123|
|         10-49 ratings|      171433|            3.943|         179.13|                    27.43|                     0.268|
|        50-199 ratings|      179101|            3.943|         176.73|                    29.64|                     0.235|
|       200-999 ratings|      162341|             4.03|         171.76|                    27.56|                     0.318|
|         1000+ ratings|       91663|            4.077|         159.54|                    22.87|                     0.403|


In [64]:
#rating polarization by product popularity
product_rating_stats = joined_df.groupBy(
    "parent_asin",
    "product_popularity_bin"
).agg(
    F.count("*").alias("review_count"),
    F.avg("rating").alias("avg_rating"),
    F.stddev("rating").alias("rating_stddev"),
    F.avg(
        F.when(F.col("rating").isin(1.0, 5.0), 1).otherwise(0)
    ).alias("extreme_rating_share")
)

In [65]:
polarization_analysis = product_rating_stats.groupBy(
    "product_popularity_bin"
).agg(
    F.count("*").alias("product_count"),
    F.round(F.avg("rating_stddev"), 3).alias("avg_rating_stddev"),
    F.round(F.avg("extreme_rating_share") * 100, 2)
        .alias("avg_extreme_rating_percent")
)

In [66]:
polarization_analysis.orderBy(
    F.when(F.col("product_popularity_bin") == "Under 10 ratings", 1)
     .when(F.col("product_popularity_bin") == "10-49 ratings", 2)
     .when(F.col("product_popularity_bin") == "50-199 ratings", 3)
     .when(F.col("product_popularity_bin") == "200-999 ratings", 4)
     .otherwise(5)
).show()

+----------------------+-------------+-----------------+--------------------------+
|product_popularity_bin|product_count|avg_rating_stddev|avg_extreme_rating_percent|
+----------------------+-------------+-----------------+--------------------------+
|      Under 10 ratings|        56854|            1.014|                     72.89|
|         10-49 ratings|        38001|            1.109|                     74.23|
|        50-199 ratings|        13014|            1.232|                     74.93|
|       200-999 ratings|         4010|            1.268|                     75.83|
|         1000+ ratings|          686|            1.249|                     75.99|
+----------------------+-------------+-----------------+--------------------------+



In [67]:
#save cleaned data as Parquet
reviews_parquet_path = "/home/jovyan/work/data/processed/reviews_clean.parquet"
metadata_parquet_path = "/home/jovyan/work/data/processed/metadata_clean.parquet"
joined_parquet_path = "/home/jovyan/work/data/processed/joined_reviews.parquet"

In [68]:
reviews_clean_df.write \
    .mode("overwrite") \
    .parquet(reviews_parquet_path)

metadata_for_join_df.write \
    .mode("overwrite") \
    .parquet(metadata_parquet_path)

joined_df.write \
    .mode("overwrite") \
    .parquet(joined_parquet_path)

In [69]:
reviews_parquet_df = spark.read.parquet(reviews_parquet_path)
metadata_parquet_df = spark.read.parquet(metadata_parquet_path)
joined_parquet_df = spark.read.parquet(joined_parquet_path)

print("Reviews rows:", reviews_parquet_df.count())
print("Metadata rows:", metadata_parquet_df.count())
print("Joined rows:", joined_parquet_df.count())

Reviews rows: 694253
Metadata rows: 112590
Joined rows: 694253


In [70]:
# record time to compare parquet vs. json
import time
from pyspark.sql import functions as F

In [71]:
raw_benchmark_parquet_path = (
    "/home/jovyan/work/data/processed/"
    "reviews_raw_benchmark.parquet"
)

raw_benchmark_df = spark.read.json(
    "/home/jovyan/work/data/raw/All_Beauty.jsonl"
).select(
    "rating",
    "verified_purchase"
)

raw_benchmark_df.write \
    .mode("overwrite") \
    .parquet(raw_benchmark_parquet_path)

In [72]:
print("JSON rows:", raw_benchmark_df.count())

print(
    "Benchmark Parquet rows:",
    spark.read.parquet(raw_benchmark_parquet_path).count()
)

JSON rows: 701528
Benchmark Parquet rows: 701528


In [73]:
import time
import statistics
from pyspark.sql import functions as F

json_path = "/home/jovyan/work/data/raw/All_Beauty.jsonl"

parquet_path = (
    "/home/jovyan/work/data/processed/"
    "reviews_raw_benchmark.parquet"
)


def run_json_benchmark():
    start = time.time()

    result = (
        spark.read.json(json_path)
        .select("rating", "verified_purchase")
        .filter(F.col("rating") >= 4)
        .groupBy("verified_purchase")
        .count()
    )

    result.collect()
    return time.time() - start


def run_parquet_benchmark():
    start = time.time()

    result = (
        spark.read.parquet(parquet_path)
        .select("rating", "verified_purchase")
        .filter(F.col("rating") >= 4)
        .groupBy("verified_purchase")
        .count()
    )

    result.collect()
    return time.time() - start

In [74]:
json_times_equal = [
    run_json_benchmark()
    for _ in range(5)
]

parquet_times_equal = [
    run_parquet_benchmark()
    for _ in range(5)
]

json_median_equal = statistics.median(json_times_equal)
parquet_median_equal = statistics.median(parquet_times_equal)

print(
    "JSON times:",
    [round(x, 3) for x in json_times_equal]
)

print(
    "Parquet times:",
    [round(x, 3) for x in parquet_times_equal]
)

print(
    "JSON median:",
    round(json_median_equal, 3),
    "seconds"
)

print(
    "Parquet median:",
    round(parquet_median_equal, 3),
    "seconds"
)

print(
    "Apples-to-apples speedup:",
    round(json_median_equal / parquet_median_equal, 2),
    "x faster"
)

JSON times: [0.747, 0.768, 0.675, 0.656, 0.607]
Parquet times: [0.187, 0.09, 0.061, 0.062, 0.062]
JSON median: 0.675 seconds
Parquet median: 0.062 seconds
Apples-to-apples speedup: 10.84 x faster
